# FER — Facial Emotion Recognition


**Input:** 6-channel stacked image (original + occluded face)  
**Models:** ResNet18 · ResNet50  · ViT  · Multi Scale ViT  

##Install & Mount

In [1]:
print("ka")

ka


In [2]:
import os
!pip install timm mediapipe --quiet

# from google.colab import drive
# drive.mount('/content/drive')

TRAIN_DIR = r'C:\Users\HomePC\Capstone_code\capstone-autism-pipeline\data\Autism_Dataset\train'
TEST_DIR = r'C:\Users\HomePC\Capstone_code\capstone-autism-pipeline\data\Autism_Dataset\test'
MODELS_DIR = r'C:\Users\HomePC\Capstone_code\capstone-autism-pipeline\data\Autism_Dataset\train\thesis_code\models'
OUTPUT_DIR = r'C:\Users\HomePC\Capstone_code\capstone-autism-pipeline\data\Autism_Dataset\train\thesis_code'


print(f'TRAIN_DIR: {TRAIN_DIR}')
print(f'TEST_DIR: {TEST_DIR}')
print(f'MODELS_DIR: {MODELS_DIR}')
print(f'OUTPUT_DIR: {OUTPUT_DIR}')


TRAIN_DIR: C:\Users\HomePC\Capstone_code\capstone-autism-pipeline\data\Autism_Dataset\train
TEST_DIR: C:\Users\HomePC\Capstone_code\capstone-autism-pipeline\data\Autism_Dataset\test
MODELS_DIR: C:\Users\HomePC\Capstone_code\capstone-autism-pipeline\data\Autism_Dataset\train\thesis_code\models
OUTPUT_DIR: C:\Users\HomePC\Capstone_code\capstone-autism-pipeline\data\Autism_Dataset\train\thesis_code


##Imports

In [3]:
import os
import json
import shutil
import random
import urllib.request
from pathlib import Path

import numpy as np
import cv2
import timm
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import seaborn as sns

from PIL import Image
from torchvision import transforms, models
from torchvision.models import efficientnet_b0
from torch.utils.data import Dataset, DataLoader, random_split

import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python.vision import face_landmarker
from mediapipe.tasks.python.vision.core.vision_task_running_mode import VisionTaskRunningMode

from sklearn.metrics import  confusion_matrix, recall_score, precision_score,f1_score, roc_curve, auc, ConfusionMatrixDisplay, classification_report
from sklearn.preprocessing import label_binarize

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

Device: cpu


##Constants

In [4]:
EMOTIONS    = ['anger', 'fear', 'joy', 'Natural', 'sadness', 'surprise']

CLASS_MAP = {}
for i, e in enumerate(EMOTIONS):
    CLASS_MAP[e] = i

IDX_CLASS = {}
for e, i in CLASS_MAP.items():
    IDX_CLASS[i] = e


NUM_CLASSES = len(EMOTIONS)

NUM_EPOCHS  = 50
BATCH_SIZE  = 32
PATIENCE    = 5

print(f'Classes : {EMOTIONS}')
print(f'Epochs  : {NUM_EPOCHS}  |  Patience: {PATIENCE}')

Classes : ['anger', 'fear', 'joy', 'Natural', 'sadness', 'surprise']
Epochs  : 50  |  Patience: 5


## Download MediaPipe Model

In [5]:
from pathlib import Path
import urllib.request

model_path = Path('models/face_landmarker.task')
model_path.parent.mkdir(parents=True, exist_ok=True)

if not model_path.exists():
    print('Downloading face_landmarker.task...')
    urllib.request.urlretrieve(
        'https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/1/face_landmarker.task',
        model_path
    )
    print('Done')
else:
    print('Model already exists')

Done


## Occlusion Class

In [6]:
class FacialRegionOcclusion:

    REGION_INDICES = {
        'left_eye':  [33, 160, 158, 133, 153, 144],
        'right_eye': [362, 385, 387, 263, 373, 380],
        'nose':      [1, 2, 98, 327, 168, 195],
    }

    def __init__(self, regions=['left_eye', 'right_eye', 'nose'],occlusion_value=0, model_path=MP_MODEL_PATH):

        self.regions  = regions
        self.occlusion_value = occlusion_value
        self.region_indices  = self.REGION_INDICES

        base_options = python.BaseOptions(model_asset_path=model_path)
        options = face_landmarker.FaceLandmarkerOptions(
            base_options=base_options,
            running_mode=VisionTaskRunningMode.IMAGE
        )
        self.landmarker = face_landmarker.FaceLandmarker.create_from_options(options)

    def get_landmarks(self, image_np):
        if image_np.dtype != np.uint8:
            image_np = image_np.astype(np.uint8)

        mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=image_np)
        results  = self.landmarker.detect(mp_image)
        if not results.face_landmarks:
            return None
        lms = results.face_landmarks[0]
        return [(int(lm.x * image_np.shape[1]), int(lm.y * image_np.shape[0])) for lm in lms]

    def occlude_region(self, image_np, landmarks, region_name):
        indices  = self.region_indices[region_name]
        points   = np.array([landmarks[i] for i in indices], dtype=np.int32)
        center_x = int(np.mean([p[0] for p in points]))
        center_y = int(np.mean([p[1] for p in points]))
        h, w     = image_np.shape[:2]
        box_size = 0.25
        box_h, box_w = int(h * box_size), int(w * box_size)
        top    = max(0, center_y - box_h // 2)
        bottom = min(h, top + box_h)
        left   = max(0, center_x - box_w // 2)
        right  = min(w, left + box_w)
        image_np[top:bottom, left:right] = int(self.occlusion_value * 255)
        return image_np

    def __call__(self, pil_image):
        image_np  = np.array(pil_image)
        landmarks = self.get_landmarks(image_np)
        if landmarks is None:
            return pil_image
        occluded = image_np.copy()
        for region in self.regions:
            occluded = self.occlude_region(occluded, landmarks, region)
        return Image.fromarray(occluded)


class EmotionOcclusionDataset(Dataset):
    def __init__(self, root_dir, occlusion_regions, preprocess=None):
        self.preprocess        = preprocess
        self.occlusion_regions = occlusion_regions
        self.samples           = []
        self.occluders = {
            region: FacialRegionOcclusion(regions=(region,))
            for region in occlusion_regions
        }
        root_dir = Path(root_dir)
        for class_name, label in CLASS_MAP.items():
            class_dir = root_dir / class_name
            if not class_dir.exists():
                print(f'Warning: {class_dir} not found, skipping.')
                continue
            for img_path in class_dir.glob('*.jpg'):
                self.samples.append((img_path, label))
        print(f'Total samples: {len(self.samples)}')

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        pil_image = Image.open(img_path).convert('RGB')
        original  = self.preprocess(pil_image)
        region    = region = self.occlusion_regions[idx % len(self.occlusion_regions)]
        occ_pil   = self.occluders[region](pil_image)
        occluded  = self.preprocess(occ_pil)
        stacked   = torch.cat([original, occluded], dim=0)   # (6, 224, 224)
        return stacked, label

print('Dataset classes defined.')

NameError: name 'MP_MODEL_PATH' is not defined

##DataLoaders

In [ ]:
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

occlusion_regions = ['left_eye', 'right_eye', 'nose']

full_train = EmotionOcclusionDataset(TRAIN_DIR, occlusion_regions, preprocess=train_transform)
test_data  = EmotionOcclusionDataset(TEST_DIR,  occlusion_regions, preprocess=val_transform)

train_size   = int(0.8 * len(full_train))
val_size     = len(full_train) - train_size
generator = torch.Generator().manual_seed(42)
train_dataset, val_dataset = random_split(full_train, [train_size, val_size], generator=generator)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader  = DataLoader(test_data,     batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print(f'Train : {len(train_dataset)}')
print(f'Val   : {len(val_dataset)}')
print(f'Test  : {len(test_data)}')

In [ ]:
def display_examples(dataset, n=2):
    fig, axes = plt.subplots(n, 2, figsize=(8, 4*n), squeeze=False)
    fig.suptitle('Original  |  Occluded', fontsize=14, fontweight='bold')

    def to_display(t):
        mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
        std  = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
        return (t * std + mean).permute(1, 2, 0).clamp(0, 1).numpy()

    for i in range(n):
        stacked, label = dataset[i]
        axes[i, 0].imshow(to_display(stacked[:3]))
        axes[i, 0].set_title(f'Original  (label={IDX_CLASS[label]})')
        axes[i, 0].axis('off')
        axes[i, 1].imshow(to_display(stacked[3:]))
        axes[i, 1].set_title('Occluded')
        axes[i, 1].axis('off')

    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, '6channel.png'), dpi=150)
    plt.show()

display_examples(full_train, n=2)

In [ ]:
def build_resnet18():
    m = models.resnet18(weights='IMAGENET1K_V1')
    old_w  = m.conv1.weight.data
    new_w  = torch.cat([old_w, old_w], dim=1)
    m.conv1 = nn.Conv2d(6, 64, kernel_size=7, stride=2, padding=3, bias=False)
    m.conv1.weight = nn.Parameter(new_w)
    m.fc    = nn.Linear(m.fc.in_features, NUM_CLASSES)
    return m.to(DEVICE)

def build_resnet50():
    m = models.resnet50(weights='IMAGENET1K_V1')
    old_w  = m.conv1.weight.data
    new_w  = torch.cat([old_w, old_w], dim=1)   # (64, 6, 7, 7)
    m.conv1 = nn.Conv2d(6, 64, kernel_size=7, stride=2, padding=3, bias=False)
    m.conv1.weight = nn.Parameter(new_w)
    m.fc    = nn.Linear(m.fc.in_features, NUM_CLASSES)
    return m.to(DEVICE)

def build_vit():
    m = timm.create_model('vit_base_patch32_224', pretrained=True, num_classes=NUM_CLASSES, in_chans=6)
    return m.to(DEVICE)


class MultiScaleViT(nn.Module):

    def __init__(self, num_classes=6, in_chans=6, pretrained=True):
        super().__init__()


        self.backbone = timm.create_model(
            'vit_base_patch16_224',
            pretrained=pretrained,
            num_classes=0,          # remove head — we want raw CLS token
            in_chans=in_chans,
            img_size=224
        )
        embed_dim = self.backbone.embed_dim   # 768 for vit_base

        # Downsample layers to create 112 and 56 scale inputs
        self.down2 = nn.AdaptiveAvgPool2d((112, 112))
        self.down4 = nn.AdaptiveAvgPool2d((56, 56))


        self.up_from_112 = nn.Upsample(size=(224, 224), mode='bilinear', align_corners=False)
        self.up_from_56  = nn.Upsample(size=(224, 224), mode='bilinear', align_corners=False)

        # Fusion MLP: 3 CLS tokens → num_classes
        self.fusion = nn.Sequential(
            nn.LayerNorm(embed_dim * 3),
            nn.Linear(embed_dim * 3, 512),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(512, num_classes)
        )

    def forward_one_scale(self, x):
        """Get CLS token from backbone for one scale."""
        return self.backbone(x)   # (B, embed_dim)

    def forward(self, x):
        # Scale 1 — full resolution 224
        feat_224 = self.forward_one_scale(x)

        # Scale 2 — half resolution: downsample to 112, upsample to 224
        x_112    = self.up_from_112(self.down2(x))
        feat_112 = self.forward_one_scale(x_112)

        # Scale 3 — quarter resolution: downsample to 56, upsample to 224
        x_56     = self.up_from_56(self.down4(x))
        feat_56  = self.forward_one_scale(x_56)

        # Concatenate all three CLS tokens → fuse
        fused = torch.cat([feat_224, feat_112, feat_56], dim=1)   # (B, 768*3)
        return self.fusion(fused)


def build_multiscale_vit():
    m = MultiScaleViT(num_classes=NUM_CLASSES, in_chans=6, pretrained=True)
    return m.to(DEVICE)

print('Model builders ready — including MultiScaleViT.')

In [ ]:
def train_model(model, model_name, lr, save_path, patience=PATIENCE):
    """
    Train with early stopping. Saves the best val-accuracy checkpoint.
    Returns (trained_model, best_val_acc, history).
    """
    criterion = nn.CrossEntropyLoss(label_smoothing=0.05)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=5, factor=0.5)

    best_val_acc      = 0.0
    epochs_no_improve = 0
    history           = []

    print(f'\n{"="*60}')
    print(f'  Training: {model_name}  |  LR: {lr}')
    print(f'{"="*60}')
    print(f'{"Epoch":<8}{"Train Loss":<14}{"Train Acc":<14}{"Val Loss":<14}{"Val Acc"}')
    print('-'*58)

    for epoch in range(1, NUM_EPOCHS + 1):
        # ── Train ────────────────────────────────────────────
        model.train()
        tr_loss, correct, total = 0.0, 0, 0
        for images, labels in train_loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            out  = model(images)                                    # single forward pass
            loss = criterion(out, labels)
            loss.backward()
            optimizer.step()
            tr_loss += loss.item() * images.size(0)
            correct += out.detach().argmax(1).eq(labels).sum().item()
            total   += labels.size(0)

        tr_loss /= total                                            # normalise
        tr_acc   = correct / total

        # ── Validate ─────────────────────────────────────────
        model.eval()
        vl_loss, correct_v, total_v = 0.0, 0, 0
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(DEVICE), labels.to(DEVICE)
                out    = model(images)
                loss   = criterion(out, labels)
                vl_loss   += loss.item() * images.size(0)
                correct_v += out.argmax(1).eq(labels).sum().item()
                total_v   += labels.size(0)
        vl_loss /= total_v
        vl_acc   = correct_v / total_v

        scheduler.step(vl_loss)
        history.append({'epoch': epoch, 'tr_loss': tr_loss, 'tr_acc': tr_acc,
                        'vl_loss': vl_loss, 'vl_acc': vl_acc})

        print(f'{epoch:<8}{tr_loss:<14.4f}{tr_acc:<14.4f}{vl_loss:<14.4f}{vl_acc:.4f}', end='')

        # ── Early stopping ───────────────────────────────────
        if vl_acc > best_val_acc:
            best_val_acc      = vl_acc
            epochs_no_improve = 0
            torch.save(model.state_dict(), save_path)
            print('  saved')
        else:
            epochs_no_improve += 1
            print()
            if epochs_no_improve >= patience:
                print(f'\n⚠ Early stopping at epoch {epoch} (no improvement for {patience} epochs)')
                break

    print(f'\nBest val acc: {best_val_acc:.4f}')
    # Load best weights back
    model.load_state_dict(torch.load(save_path, map_location=DEVICE))
    return model, best_val_acc, history

In [ ]:
def evaluate_model(model, loader, model_name='model'):
    """Full evaluation: accuracy, F1, ROC, confusion matrix."""
    model.eval()
    criterion = nn.CrossEntropyLoss()

    total_loss, correct, total = 0.0, 0, 0
    all_labels, all_preds, all_probs = [], [], []

    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            out        = model(images)
            probs      = torch.softmax(out, dim=1)
            total_loss += criterion(out, labels).item() * images.size(0)  # weighted sum
            preds      = out.argmax(1)
            correct    += preds.eq(labels).sum().item()
            total      += labels.size(0)
            all_labels.extend(labels.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())

    all_labels = np.array(all_labels)
    all_preds  = np.array(all_preds)
    all_probs  = np.array(all_probs)

    total_loss /= total                                                    # normalise
    acc        = correct / total
    precision  = precision_score(all_labels, all_preds, average='macro', zero_division=0)
    recall     = recall_score(all_labels, all_preds, average='macro', zero_division=0)
    f1         = f1_score(all_labels, all_preds, average='macro', zero_division=0)
    cm         = confusion_matrix(all_labels, all_preds)

    print(f'\n── {model_name} ─────────────────────────────')
    print(f'  Loss      : {total_loss:.4f}')                              # now printed
    print(f'  Accuracy  : {acc:.4f}')
    print(f'  Precision : {precision:.4f}')
    print(f'  Recall    : {recall:.4f}')
    print(f'  F1-Score  : {f1:.4f}')
    print(classification_report(all_labels, all_preds, target_names=EMOTIONS, zero_division=0))

    # Plots
    y_bin = label_binarize(all_labels, classes=list(range(NUM_CLASSES)))
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle(model_name, fontsize=13, fontweight='bold')

    for c in range(NUM_CLASSES):
        fpr, tpr, _ = roc_curve(y_bin[:, c], all_probs[:, c])
        axes[0].plot(fpr, tpr, lw=2, label=f'{EMOTIONS[c]} (AUC={auc(fpr, tpr):.2f})')
    axes[0].plot([0, 1], [0, 1], 'k--', lw=1)
    axes[0].set(title='ROC Curves', xlabel='FPR', ylabel='TPR')
    axes[0].legend(fontsize=8)

    ConfusionMatrixDisplay(cm, display_labels=EMOTIONS).plot(ax=axes[1], colorbar=True, xticks_rotation=45)
    axes[1].set_title('Confusion Matrix')

    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, f'{model_name}_evaluation.png'), dpi=150)
    plt.show()

    return acc, f1, all_probs

print('evaluate_model function defined.')

##Training Curve PLot

In [ ]:
def plot_history(history, model_name):
    epochs   = [h['epoch']   for h in history]
    tr_acc   = [h['tr_acc']  for h in history]
    vl_acc   = [h['vl_acc']  for h in history]
    tr_loss  = [h['tr_loss'] for h in history]
    vl_loss  = [h['vl_loss'] for h in history]

    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    axes[0].plot(epochs, tr_acc,  label='Train', color='#4472C4')
    axes[0].plot(epochs, vl_acc,  label='Val',   color='#ED7D31')
    axes[0].set(title='Accuracy', xlabel='Epoch', ylabel='Acc')
    axes[0].legend(); axes[0].grid(alpha=0.3)

    axes[1].plot(epochs, tr_loss, label='Train', color='#4472C4')
    axes[1].plot(epochs, vl_loss, label='Val',   color='#ED7D31')
    axes[1].set(title='Loss', xlabel='Epoch', ylabel='Loss')
    axes[1].legend(); axes[1].grid(alpha=0.3)

    plt.suptitle(f'{model_name} — Training Curves', fontweight='bold')
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, f'{model_name}_curves.png'), dpi=150)
    plt.show()

print('plot_history defined.')

In [ ]:
results   = {}
histories = {}

In [ ]:
model_resnet18, best_acc, hist = train_model(
    build_resnet18(), 'resnet18', lr=1e-3,
    save_path=os.path.join(MODELS_DIR, 'resnet18_best.pth')
)
plot_history(hist, 'resnet18')
acc, f1, _ = evaluate_model(model_resnet18, test_loader, 'resnet18')
results['resnet18']   = acc
histories['resnet18'] = hist

In [ ]:
model_resnet50, best_acc, hist = train_model(
    build_resnet50(), 'resnet50', lr = 5e-4,
    save_path=os.path.join(MODELS_DIR, 'resnet50_best.pth')
)
plot_history(hist, 'resnet50')
acc, f1, _ = evaluate_model(model_resnet50, test_loader, 'resnet50')
results['resnet50']   = acc
histories['resnet50'] = hist

In [ ]:
model_vit, best_acc, hist = train_model(
    build_vit(), 'vit', lr=5e-5,
    save_path=os.path.join(MODELS_DIR, 'vit_best.pth')
)
plot_history(hist, 'vit')
acc, f1, _ = evaluate_model(model_vit, test_loader, 'vit')
results['vit']   = acc
histories['vit'] = hist

In [ ]:
model_ms_vit, best_acc, hist = train_model(
    build_multiscale_vit(), 'multiscale_vit', lr=2e-5,
    save_path=os.path.join(MODELS_DIR, 'multiscale_vit_best.pth')
)
plot_history(hist, 'multiscale_vit')
acc, f1, _ = evaluate_model(model_ms_vit, test_loader, 'multiscale_vit')
results['multiscale_vit']   = acc
histories['multiscale_vit'] = hist

In [ ]:
print('\n' + '='*45)
print('FER MODEL COMPARISON')
print('='*45)
print(f"Current results dictionary: {results}")

if not results:
    print('Warning: Results dictionary is empty. Ensure all model training cells have been executed successfully.')
    # Exit or handle if results is empty

else:
    print(f'{ "Model":<16}  {"Test Acc":>9}')
    print('-'*30)
    for name, acc in sorted(results.items(), key=lambda x: -x[1]):
        marker = '  BEST' if acc == max(results.values()) else ''
        print(f'  {name:<14}  {acc:.4f}{marker}')

    best_fer_model_name = max(results, key=results.get)
    print(f'\n Best FER model: {best_fer_model_name}  ({results[best_fer_model_name]:.4f})')

    # Bar chart
    names   = list(results.keys())
    accs    = [results[n] for n in names]
    colours = ['#ED7D31' if n == best_fer_model_name else '#4472C4' for n in names]
    fig, ax = plt.subplots(figsize=(9, 4))
    bars = ax.bar(names, accs, color=colours, edgecolor='white')
    ax.set_ylim(0, 1.1)
    ax.set_ylabel('Test Accuracy')
    ax.set_title('FER Model Comparison', fontweight='bold')
    ax.axhline(1/NUM_CLASSES, color='grey', linestyle='--', label=f'Chance ({1/NUM_CLASSES:.2f})')
    ax.legend()
    for bar, acc in zip(bars, accs):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height()+0.01,
                f'{acc:.3f}', ha='center', fontsize=9, fontweight='bold')
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, 'fer_model_comparison.png'), dpi=150)
    plt.show()

In [ ]:
model_files = {
    'resnet18'   : os.path.join(MODELS_DIR, 'resnet18_best.pth'),
    'resnet50'   : os.path.join(MODELS_DIR, 'resnet50_best.pth'),
    'vit'        : os.path.join(MODELS_DIR, 'vit_best.pth'),
    'multiscale_vit': os.path.join(MODELS_DIR, 'multiscale_vit_best.pth'),
}

# Copy best model weights to fer_best.pth
best_src  = model_files[best_fer_model_name]
best_dest = os.path.join(MODELS_DIR, 'fer_best.pth')
shutil.copy(best_src, best_dest)

# Save config JSON — the fusion notebook reads this
fer_config = {
    'best_model'   : best_fer_model_name,
    'best_test_acc': float(results[best_fer_model_name]),
    'num_classes'  : NUM_CLASSES,
    'emotions'     : EMOTIONS,
    'class_map'    : CLASS_MAP,
    'input_channels': 6,
    'input_size'   : 224,
    'all_results'  : {k: float(v) for k, v in results.items()},
    'normalisation': {
        'mean': [0.485, 0.456, 0.406],
        'std' : [0.229, 0.224, 0.225]
    }
}

config_path = os.path.join(MODELS_DIR, 'fer_model_config.json')
with open(config_path, 'w') as f:
    json.dump(fer_config, f, indent=2)

print(f' Best FER model : {best_fer_model_name}  ({results[best_fer_model_name]:.4f})')
print(f' Weights saved  : {best_dest}')
print(f'Config saved   : {config_path}')
print(f'\nAll models in: {MODELS_DIR}')
for f in sorted(os.listdir(MODELS_DIR)):
    print(f'  {f}')

In [ ]:
def format_fer_output(logits, emotions):
    probs = torch.softmax(logits, dim=1).cpu().numpy()[0]

    emotion_scores = [
        {
            "label": emotions[i].capitalize(),
            "value": round(float(probs[i] * 100), 2)
        }
        for i in range(len(probs))
    ]

    top_idx = int(np.argmax(probs))

    result = {
        "top_emotion": emotions[top_idx].capitalize(),
        "confidence": round(float(probs[top_idx] * 100), 2),
        "emotions": emotion_scores
    }

    return result